# 실패 실행에서 평가·배포 판단까지

시험 요청 하나에서 발견한 실패를 개발 데이터에 추가하고 후보 버전에서 고쳐졌는지 검증한다.

### 노트북 실행 준비

Python 3.12 커널에서 위쪽 셀부터 실행하세요. 이 예제는 파이썬 표준 라이브러리로 실행할 수 있습니다.

## 실행 하나가 평가 문항이 되기까지

느렸다는 사용자 피드백만으로는 정답을 정할 수 없다. 해당 실행의 입력과 당시 유효한 원문 조항을 찾아 기대 행동을 붙인다. 개발 질문으로 추가한 뒤 최종 확인용 질문은 따로 보관한다.

도구 경로에는 승인 받기 전 확정처럼 품질 평균과 별도로 막아야 할 실패가 있다. 검색 정답률이 올라가더라도 이런 필수 사례가 깨지면 배포 후보를 보류할 수 있다.

아래 비교표는 가상 실험 출력이다. 수업 서비스에서는 LangSmith 실험의 버전·실행 ID를 함께 기록해 같은 질문으로 돌아갈 수 있게 한다.

## 평균이 같아도 보류할 수 있는 후보

기준선과 후보가 같은 세 질문에 답했다. Q3은 사용자의 승인이 먼저여야 한다.

In [1]:
base = {"Q1":True,"Q2":False,"Q3":True}
candidate = {"Q1":True,"Q2":True,"Q3":False}
required = {"Q3"}
def compare(base, candidate, required):
    if base.keys() != candidate.keys():
        raise ValueError("질문 집합 불일치")
    regressions = [q for q in base if base[q] and not candidate[q]]
    release = all(candidate[q] for q in required)
    return regressions, release

print(compare(base,candidate,required))

(['Q3'], False)


Q2가 고쳐지고 Q3가 새로 실패했다. 전체 정답 수만 보면 둘 다 2건이지만 승인 경로는 통과하지 않았다. 여기서 release는 예시의 필수 사례 조건만 판정한다.

### 사라진 평가 문항 감지하기

후보 실행에서 Q3가 빠졌다. 공통 문항만 골라 점수를 높게 계산하지 않고 평가 자체를 중단해야 한다.

In [2]:
partial = {k:v for k,v in candidate.items() if k != "Q3"}

In [3]:
# 이 셀에 풀이를 작성하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre><code>try:
    compare(base, partial, required)
except ValueError as error:
    print(error)</code></pre><pre>질문 집합 불일치
</pre><p>누락된 질문을 오답으로 처리할지 실험 실패로 기록할지는 팀 정책이지만, 평가 대상이 달랐다는 사실은 먼저 밝혀야 한다.</p></details>

## 같은 실패 실행을 두 번 수집하면?

사용자가 버튼을 두 번 누르는 바람에 같은 실패 피드백이 두 건 들어왔다. 실행 ID와 기대 답을 따로 보관하고, 이미 수집한 실행은 평가에서 한 번만 센다.

In [4]:
events = [("r8","Q-revision","A2"), ("r8","Q-revision","A2"), ("r9","Q-empty",None)]
def to_examples(events):
    seen, examples = set(), []
    for run, question, expected in events:
        if run in seen:
            continue
        examples.append({"run":run,"question":question,"expected":expected})
        seen.add(run)
    return examples
print([r["run"] for r in to_examples(events)])

['r8', 'r9']


실행은 r8과 r9 두 건이다. 기대 근거가 None인 Q-empty는 자료가 없어 보류해야 하는 문항으로 보존한다.

### 확인용 질문이 개발 세트에 들어왔는지 찾기

회귀 세트를 받아 최종 확인 질문과 겹치는 ID를 검사한다. 같은 실행 중복을 제거한 뒤에도 데이터 분리 문제는 남을 수 있다.

In [5]:
held_out = {"Q-empty", "Q-unseen"}

In [6]:
# 이 셀에 풀이를 작성하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre><code>dev = {r[&quot;question&quot;] for r in to_examples(events)}
print(sorted(dev &amp; held_out))</code></pre><pre>[&#x27;Q-empty&#x27;]
</pre><p>Q-empty가 겹친다. 이 사례로 수정했다면 최종 성능을 판정할 별도 질문이 필요하다.</p></details>

## 다른 입력에 적용하기

시험 배포에서 실패 실행 하나를 고르고 기대 조항과 호출 순서를 찾아 개발 데이터에 등록한다. 같은 사례의 중복 등록을 막고, 고친 뒤에는 새로 실패한 질문과 비용·지연 변화도 비교한다.

마지막 확인 질문은 수정이 끝난 뒤에 열어 최종 판정을 기록한다. 모델·프롬프트·색인·질문 버전을 바꾼 실험끼리 섞지 않는다.